### 1. Environment (grid world with dirt)

In [ ]:
import random
from collections import deque

ACTIONS = {"UP": (-1, 0), "DOWN": (1, 0), "LEFT": (0, -1), "RIGHT": (0, 1)}


class GridWorld:

    def __init__(self, layout: list[str]):
        self.walls, self.dirt = set(), set()
        for r, row in enumerate(layout):
            for c, ch in enumerate(row):
                if ch == "#":
                    self.walls.add((r, c))
                elif ch == "*":
                    self.dirt.add((r, c))
                elif ch == "A":
                    self.agent = (r, c)
        self.rows, self.cols = len(layout), len(layout[0])
        self.total_dirt = len(self.dirt)

    def is_open(self, pos) -> bool:
        r, c = pos
        return 0 <= r < self.rows and 0 <= c < self.cols and pos not in self.walls

    def percept(self) -> dict:
        # local view only: what the agent can sense from its own cell
        r, c = self.agent
        neighbours = {a: (r + dr, c + dc) for a, (dr, dc) in ACTIONS.items()}
        return {
            "dirty": self.agent in self.dirt,
            "open": [a for a, p in neighbours.items() if self.is_open(p)],
            "dirty_near": [a for a, p in neighbours.items() if p in self.dirt],
        }

    def step(self, action: str) -> None:
        if action == "SUCK":
            self.dirt.discard(self.agent)
            return
        dr, dc = ACTIONS[action]
        nxt = (self.agent[0] + dr, self.agent[1] + dc)
        if self.is_open(nxt):
            self.agent = nxt

    def render(self) -> str:
        out = []
        for r in range(self.rows):
            row = ""
            for c in range(self.cols):
                p = (r, c)
                row += (
                    "A"
                    if p == self.agent
                    else "#" if p in self.walls else "*" if p in self.dirt else "."
                )
            out.append(row)
        return "\n".join(out)


LAYOUT = [
    "A...#...",
    ".##.#.*.",
    ".*......",
    "...##.#.",
    ".#..*.#.",
    "...#..*.",
]

print(GridWorld(LAYOUT).render())

A...#...
.##.#.*.
.*......
...##.#.
.#..*.#.
...#..*.


### 2. Reactive agent (stimulus -> response rules, no memory, no goal)

In [2]:
class ReactiveAgent:
    """Maps the current percept straight to an action using condition-action rules."""

    def __init__(self, seed: int = 0):
        self.rng = random.Random(seed)

    def act(self, percept: dict, env: GridWorld = None) -> str:
        if percept["dirty"]:
            return "SUCK"
        if percept["dirty_near"]:
            return percept["dirty_near"][0]
        if not percept["open"]:
            return "NOOP"  # boxed in by walls
        return self.rng.choice(percept["open"])  # nothing sensed, wander

### 3. Goal-oriented agent (plans actions to reach the goal: all dirt cleaned)

In [3]:
class GoalAgent:
    """Knows the map, checks the goal state and plans paths with BFS to reach it."""

    def __init__(self):
        self.plan: list[str] = []

    @staticmethod
    def goal_reached(env: GridWorld) -> bool:
        return not env.dirt

    def bfs(self, env: GridWorld, start, targets: set) -> list[str]:
        # shortest action sequence from start to the nearest target cell
        frontier, parent = deque([start]), {start: None}
        while frontier:
            cur = frontier.popleft()
            if cur in targets:
                path = []
                while parent[cur]:
                    cur, action = parent[cur]
                    path.append(action)
                return path[::-1]
            for action, (dr, dc) in ACTIONS.items():
                nxt = (cur[0] + dr, cur[1] + dc)
                if env.is_open(nxt) and nxt not in parent:
                    parent[nxt] = (cur, action)
                    frontier.append(nxt)
        return []  # remaining dirt unreachable

    def act(self, percept: dict, env: GridWorld) -> str:
        if self.goal_reached(env):
            return "NOOP"
        if percept["dirty"]:
            return "SUCK"
        if not self.plan:
            self.plan = self.bfs(env, env.agent, env.dirt)
            if not self.plan:
                return "NOOP"
        return self.plan.pop(0)

### 4. Run loop

In [4]:
def run(agent, layout: list[str], max_steps: int = 100, verbose: bool = False) -> dict:
    env = GridWorld(layout)
    trace = []
    for step in range(1, max_steps + 1):
        action = agent.act(env.percept(), env)
        if action == "NOOP":
            step -= 1
            break
        env.step(action)
        trace.append(action)
        if not env.dirt:
            break

    if verbose:
        print(" -> ".join(trace))
        print(f"\nfinal grid after {step} steps:\n{env.render()}")

    return {
        "steps": step,
        "cleaned": env.total_dirt - len(env.dirt),
        "total": env.total_dirt,
        "success": not env.dirt,
    }

### 5. Single run on the same layout

In [5]:
print("REACTIVE AGENT")
reactive_result = run(ReactiveAgent(seed=1), LAYOUT, verbose=True)
print(reactive_result)

print("\nGOAL-ORIENTED AGENT")
goal_result = run(GoalAgent(), LAYOUT, verbose=True)
print(goal_result)

REACTIVE AGENT
DOWN -> UP -> RIGHT -> LEFT -> RIGHT -> RIGHT -> RIGHT -> LEFT -> LEFT -> LEFT -> RIGHT -> LEFT -> RIGHT -> RIGHT -> LEFT -> RIGHT -> RIGHT -> DOWN -> UP -> LEFT -> LEFT -> LEFT -> DOWN -> UP -> RIGHT -> LEFT -> RIGHT -> LEFT -> DOWN -> DOWN -> RIGHT -> SUCK -> LEFT -> RIGHT -> DOWN -> LEFT -> UP -> RIGHT -> DOWN -> LEFT -> DOWN -> UP -> DOWN -> UP -> UP -> RIGHT -> RIGHT -> LEFT -> DOWN -> RIGHT -> DOWN -> RIGHT -> RIGHT -> SUCK -> RIGHT -> LEFT -> LEFT -> LEFT -> DOWN -> LEFT -> RIGHT -> LEFT -> LEFT -> RIGHT -> LEFT -> RIGHT -> RIGHT -> UP -> DOWN -> LEFT -> LEFT -> RIGHT -> LEFT -> UP -> DOWN -> RIGHT -> RIGHT -> UP -> DOWN -> UP -> DOWN -> LEFT -> LEFT -> UP -> UP -> UP -> UP -> UP -> RIGHT -> RIGHT -> RIGHT -> LEFT -> RIGHT -> DOWN -> DOWN -> RIGHT -> LEFT -> RIGHT -> LEFT -> LEFT

final grid after 100 steps:
....#...
.##.#.*.
..A.....
...##.#.
.#....#.
...#..*.
{'steps': 100, 'cleaned': 2, 'total': 4, 'success': False}

GOAL-ORIENTED AGENT
DOWN -> DOWN -> RIGHT ->

### 6. Compare over many random environments

In [6]:
def random_layout(rng: random.Random, rows=6, cols=8, walls=8, dirt=5) -> list[str]:
    cells = [(r, c) for r in range(rows) for c in range(cols)]
    rng.shuffle(cells)
    grid = [["."] * cols for _ in range(rows)]
    ar, ac = cells[0]
    grid[ar][ac] = "A"
    for r, c in cells[1 : 1 + walls]:
        grid[r][c] = "#"
    for r, c in cells[1 + walls : 1 + walls + dirt]:
        grid[r][c] = "*"
    return ["".join(row) for row in grid]


rng = random.Random(42)
layouts = [random_layout(rng) for _ in range(200)]

summary = {}
for name, make_agent in {
    "reactive": lambda i: ReactiveAgent(seed=i),
    "goal_oriented": lambda i: GoalAgent(),
}.items():
    results = [run(make_agent(i), lay) for i, lay in enumerate(layouts)]
    wins = [r for r in results if r["success"]]
    summary[name] = {
        "success_rate": len(wins) / len(results),
        "avg_dirt_cleaned": sum(r["cleaned"] / r["total"] for r in results) / len(results),
        "avg_steps_when_success": sum(r["steps"] for r in wins) / max(len(wins), 1),
    }

print(f"{'agent':<15}{'success rate':>14}{'dirt cleaned':>14}{'avg steps':>12}")
for name, s in summary.items():
    print(
        f"{name:<15}{s['success_rate']:>14.0%}{s['avg_dirt_cleaned']:>14.0%}"
        f"{s['avg_steps_when_success']:>12.1f}"
    )

agent            success rate  dirt cleaned   avg steps
reactive                  33%           78%        61.5
goal_oriented             94%           99%        21.6


### 7. Observations

- **Reactive agent:** reacts only to what it senses right now (dirt here / next to it) using fixed condition-action rules. It is fast to compute and needs no model of the world, but with nothing in view it wanders randomly, revisits cells and often runs out of steps before the room is clean.
- **Goal-oriented agent:** keeps an explicit goal (no dirt left), knows the map and uses BFS to plan the shortest path to the nearest dirt. It cleans every reachable dirt cell in far fewer steps and stops as soon as the goal test passes. Its few failures are layouts where random walls fully enclose a dirt cell, so BFS finds no path and the goal is unreachable.
- Both agents use the same rule "if dirty, suck", the difference is what happens when there is no immediate stimulus: random wandering vs. a planned path toward the goal.
- Tradeoff: the goal-oriented agent needs a world model (map, dirt locations) and search time per plan, while the reactive agent works with only local sensors and constant-time decisions, so it suits simple or fully local tasks.